# Heart Disease Prediction for Beginners

Hi! In this notebook I am trying to predict whether a person has heart disease or not based on simple questions about their lifestyle and health habits.

I am using a real dataset of people answering questions like how much they sleep, if they smoke, and how they rate their general health.

**Goal:** Predict if someone has heart disease (Yes/No) based on simple questions.

## Importing Libraries

First I will import all the libraries that I need to work with the data and visualize it.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
import pickle
import os

## 1. Loading the Data

Let me load the dataset from the CSV file and look at the first few rows.

In [ ]:
df = pd.read_csv('heart_2020_cleaned.csv')
print('Shape of data:', df.shape)
print('Missing values:', df.isnull().sum().sum())
df.head()

# checking how many people have heart disease vs don't have
print('Target distribution:')
print(df['HeartDisease'].value_counts())
print()
print('No heart disease:', (df['HeartDisease']=='No').sum())
print('Heart disease:', (df['HeartDisease']=='Yes').sum())

## 2. Choosing Important Features

Our dataset has many columns, but I only want to keep the ones that are easy for a normal person to answer.

I will use features like:
- `BMI`: Body Mass Index
- `Smoking`: Do you smoke?
- `AlcoholDrinking`: Do you drink alcohol?
- `Stroke`: Have you had a stroke?
- `PhysicalHealth`: Days of poor physical health
- `DiffWalking`: Difficulty walking
- `Sex`: Gender
- `AgeCategory`: Age group
- `Diabetic`: Are you diabetic?
- `PhysicalActivity`: Are you physically active?
- `GenHealth`: General health rating
- `SleepTime`: Hours of sleep per night
- `KidneyDisease`: History of kidney disease

In [ ]:
features_raw = ['BMI', 'Smoking', 'AlcoholDrinking', 'Stroke', 'PhysicalHealth',
               'DiffWalking', 'Sex', 'AgeCategory', 'Diabetic', 'PhysicalActivity',
               'GenHealth', 'SleepTime', 'KidneyDisease']

target = 'HeartDisease'

# Create a working copy of the dataframe
df_work = df[features_raw + [target]].copy()

# Convert Yes/No heart disease to 1 and 0 so the machine can understand it
df_work[target] = (df_work[target] == 'Yes').astype(int)

print('Target after changing to numbers:')
print(df_work[target].value_counts())

## 3. Changing Text to Numbers

Machine learning models only understand numbers, so I need to change text answers like 'Yes'/'No' or 'Male'/'Female' into numbers.

In [ ]:
# 1. Change simple Yes/No answers to 1 and 0
binary_cols = ['Smoking', 'AlcoholDrinking', 'Stroke', 'DiffWalking', 'PhysicalActivity', 'KidneyDisease']
for col in binary_cols:
    df_work[col] = (df_work[col] == 'Yes').astype(int)

# 2. Gender: Make Male=1, Female=0
df_work['Sex'] = (df_work['Sex'] == 'Male').astype(int)

# 3. Age Groups: Give a number from 1 to 13 depending on how old they are
age_order = ['18-24', '25-29', '30-34', '35-39', '40-44', '45-49',
             '50-54', '55-59', '60-64', '65-69', '70-74', '75-79', '80 or older']
age_map = {v: i+1 for i, v in enumerate(age_order)}
df_work['AgeCategory'] = df_work['AgeCategory'].map(age_map)

# 4. Diabetic: Simplify to 1 (Yes) and 0 (No)
df_work['Diabetic'] = df_work['Diabetic'].isin(['Yes','Yes (during pregnancy)']).astype(int)

# 5. General Health: Rate from 1 (Poor) to 5 (Excellent)
gen_map = {'Poor': 1, 'Fair': 2, 'Good': 3, 'Very good': 4, 'Excellent': 5}
df_work['GenHealth'] = df_work['GenHealth'].map(gen_map)

print('Done converting text to numbers!')
df_work.head()

features = features_raw

# Let's see which features are most strongly related to heart disease
print('Correlation with Heart Disease:')
corr = df_work[features].corrwith(df_work[target]).abs().sort_values(ascending=False)
print(corr.round(3))

## 4. Exploratory Data Analysis (EDA)

Now I will make some graphs to visually understand the data and see what factors are most common for people with heart disease.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Heart Disease - Lifestyle Feature Analysis', fontsize=14, fontweight='bold')

# plot 1 - Class balance
df_work[target].value_counts().plot(kind='bar', ax=axes[0,0],
    color=['mediumseagreen','tomato'], edgecolor='black', rot=0)
axes[0,0].set_xticklabels(['No Disease','Heart Disease'])
axes[0,0].set_title('Class Distribution')

# plot 2 - BMI
df_work[df_work[target]==0]['BMI'].hist(bins=40, ax=axes[0,1], alpha=0.7, color='mediumseagreen', label='No')
df_work[df_work[target]==1]['BMI'].hist(bins=40, ax=axes[0,1], alpha=0.7, color='tomato', label='Yes')
axes[0,1].set_title('BMI by Outcome')
axes[0,1].set_xlabel('BMI')
axes[0,1].set_xlim(10, 70)
axes[0,1].legend()

# plot 3 - Age
df_work[df_work[target]==0]['AgeCategory'].hist(bins=13, ax=axes[0,2], alpha=0.7, color='mediumseagreen', label='No')
df_work[df_work[target]==1]['AgeCategory'].hist(bins=13, ax=axes[0,2], alpha=0.7, color='tomato', label='Yes')
axes[0,2].set_title('Age Group by Outcome')
axes[0,2].legend()

# plot 4 - Heart disease rate by GenHealth
gen_rate = df_work.groupby('GenHealth')[target].mean()
axes[1,0].bar(gen_rate.index, gen_rate.values, color='tomato', edgecolor='k')
axes[1,0].set_title('Heart Disease Rate by General Health')
axes[1,0].set_xlabel('General Health (1=Poor, 5=Excellent)')
axes[1,0].set_ylabel('Heart Disease Rate')

# plot 5 - Risk factors comparison
binary_show = ['Smoking','AlcoholDrinking','Stroke','DiffWalking','Diabetic','KidneyDisease']
disease_rates = [df_work[df_work[target]==1][f].mean() for f in binary_show]
healthy_rates = [df_work[df_work[target]==0][f].mean() for f in binary_show]
x = np.arange(len(binary_show))
axes[1,1].bar(x-0.18, healthy_rates, 0.35, label='No Disease', color='mediumseagreen', edgecolor='k')
axes[1,1].bar(x+0.18, disease_rates, 0.35, label='Heart Disease', color='tomato', edgecolor='k')
axes[1,1].set_xticks(x)
axes[1,1].set_xticklabels(binary_show, rotation=25, ha='right', fontsize=8)
axes[1,1].set_title('Risk Factor Rates by Outcome')
axes[1,1].legend()

# plot 6 - Correlation bar chart
corr.sort_values().plot(kind='barh', ax=axes[1,2], color='tomato', edgecolor='k')
axes[1,2].set_title('Feature Correlation with Heart Disease')

plt.tight_layout()
plt.savefig('heart_eda.png', bbox_inches='tight')
plt.show()
print("EDA plot saved.")

## 5. Preparing Data for Machine Learning

The dataset has way too many healthy people compared to people with heart disease. If I teach the model using this, it will just learn to always guess "No Disease".

So, I will balance it out so the computer sees an equal number of both. Then, I will split the data into a training set and testing set.

In [ ]:
df_majority = df_work[df_work[target]==0]
df_minority = df_work[df_work[target]==1]

print('Before balancing:')
print('  No heart disease:', len(df_majority))
print('  Heart disease:', len(df_minority))

# undersampling the healthy ones
df_majority_down = resample(df_majority, replace=False,
                            n_samples=len(df_minority), random_state=42)
df_balanced = pd.concat([df_majority_down, df_minority]).sample(frac=1, random_state=42)

print()
print('After balancing:')
print(df_balanced[target].value_counts())

# Now split into features (X) and target (y)
X = df_balanced[features]
y = df_balanced[target]

# Split 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# We scale the features so numbers like BMI (20-40) and SleepTime (0-24) don't confuse the model
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print()
print('Training samples (for learning):', X_train.shape[0])
print('Testing samples (for checking):', X_test.shape[0])

## 5. Training the Model

Now I will train a simple Logistic Regression model using the training data so we can save it.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# create and train the model
final_model = LogisticRegression(max_iter=1000)
final_model.fit(X_train_sc, y_train)

# test it
predictions = final_model.predict(X_test_sc)
accuracy = accuracy_score(y_test, predictions)

print(f'Model Accuracy: {accuracy * 100:.2f}%')

## 6. Saving the Model

Finally, I will save the trained model, the scaler, and the translation dictionaries for age and health ratings. This way, I can load them up later in a separate web application without running all this code again.

In [ ]:
os.makedirs('models', exist_ok=True)

pickle.dump(final_model, open('models/heart_model.pkl',    'wb'))
pickle.dump(scaler,      open('models/heart_scaler.pkl',   'wb'))
pickle.dump(features,    open('models/heart_features.pkl', 'wb'))
pickle.dump(age_map,     open('models/heart_age_map.pkl',  'wb'))
pickle.dump(gen_map,     open('models/heart_gen_map.pkl',  'wb'))

print('Saved successfully!')
print()
print('Web app inputs for heart disease will need:')
for f in features:
    print(' -', f)